# 🌳 Capítulo 8 — Árboles (Trees)
## Clase 1 — Ejercicios Prácticos: Fundamentos

**Libro:** Goodrich, Tamassia & Goldwasser — §8.1–§8.4  
**Materia:** Estructuras de Datos y Algoritmos — Lic. en Sistemas | **Año:** 2026

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/CristianPacifico/prog3-ls-fcad-uner/blob/main/cap08/Ejercicios_Clase1_Fundamentos.ipynb)

## 🎯 Objetivos

Luego de esta clase práctica podrás:
1. Calcular profundidad, altura y propiedades de un árbol a mano.
2. Implementar `depth()` y `height()` de forma recursiva.
3. Simular recorridos preorden, postorden, BFS e inorden sobre un árbol dado.
4. Implementar BFS usando una cola.
5. Contar nodos hoja/internos y verificar propiedades de árboles binarios.

## 🔧 Configuración — Importaciones y clases base

In [ ]:
from abc import ABCMeta, abstractmethod
from collections import deque

# ── Tree ABC (base) ───────────────────────────────────────────────────────────
class Tree(metaclass=ABCMeta):
    class Position(metaclass=ABCMeta):
        @abstractmethod
        def element(self): raise NotImplementedError
        @abstractmethod
        def __eq__(self, other): raise NotImplementedError
        def __ne__(self, other): return not (self == other)
    @abstractmethod
    def root(self): raise NotImplementedError
    @abstractmethod
    def parent(self, p): raise NotImplementedError
    @abstractmethod
    def num_children(self, p): raise NotImplementedError
    @abstractmethod
    def children(self, p): raise NotImplementedError
    @abstractmethod
    def __len__(self): raise NotImplementedError
    def is_root(self, p): return self.root() == p
    def is_leaf(self, p): return self.num_children(p) == 0
    def is_empty(self): return len(self) == 0
    def positions(self):
        if not self.is_empty():
            q = deque([self.root()])
            while q:
                p = q.popleft(); yield p
                for c in self.children(p): q.append(c)

# Árbol general simple para demos
class SimpleTree(Tree):
    class _Node:
        __slots__ = '_e','_parent','_children'
        def __init__(self, e, parent=None):
            self._e=e; self._parent=parent; self._children=[]
    class _Position(Tree.Position):
        def __init__(self,c,n): self._container=c; self._node=n
        def element(self): return self._node._e
        def __eq__(self,other): return type(other) is type(self) and other._node is self._node
    def _validate(self,p):
        if not isinstance(p,self._Position): raise TypeError
        if p._container is not self: raise ValueError
        return p._node
    def _make_position(self,node): return self._Position(self,node) if node else None
    def __init__(self): self._root=None; self._size=0
    def __len__(self): return self._size
    def root(self): return self._make_position(self._root)
    def parent(self,p): return self._make_position(self._validate(p)._parent)
    def num_children(self,p): return len(self._validate(p)._children)
    def children(self,p):
        for n in self._validate(p)._children: yield self._make_position(n)
    def add_root(self,e):
        self._root=self._Node(e); self._size=1; return self._make_position(self._root)
    def add_child(self,p,e):
        n=self._validate(p); child=self._Node(e,parent=n)
        n._children.append(child); self._size+=1; return self._make_position(child)

print("Entorno listo ✓")

---

## 📝 Ejercicio 1: Análisis manual de un árbol

Dado el siguiente árbol general:
```
           A         ← raíz
          /|\
         B C D
        /\   |\
       E  F  G H
      /
     I
```
Responde las preguntas en la celda siguiente.

In [ ]:
# Ejercicio 1 — Construir y analizar el árbol del diagrama
T = SimpleTree()
A = T.add_root('A')
B = T.add_child(A, 'B')
C = T.add_child(A, 'C')
D = T.add_child(A, 'D')
E = T.add_child(B, 'E')
F = T.add_child(B, 'F')
G = T.add_child(D, 'G')
H = T.add_child(D, 'H')
I = T.add_child(E, 'I')

# ── Preguntas ──────────────────────────────────────────────────────────────────
print("=== Análisis del Árbol ===")
print(f"Tamaño (n):              {len(T)}")
print(f"Raíz:                    {T.root().element()}")
print(f"Hijos de A:              {[c.element() for c in T.children(A)]}")
print(f"Hojas:                   {[p.element() for p in T.positions() if T.is_leaf(p)]}")
print(f"Nodos internos:          {[p.element() for p in T.positions() if not T.is_leaf(p)]}")

# EJERCICIO: Implementar depth() sin ver el código del NB01
def mi_depth(T, p):
    """Retorna la profundidad de p en T.
    COMPLETAR: usar recursión con la definición
    profundidad(raíz) = 0
    profundidad(p) = 1 + profundidad(padre(p))
    """
    # TODO: implementar
    if T.is_root(p):
        return 0
    else:
        return 1 + mi_depth(T, T.parent(p))

print()
print("=== Profundidades ===")
for nodo, pos in [('A',A),('B',B),('E',E),('I',I)]:
    print(f"  depth({nodo}) = {mi_depth(T, pos)}")

# RESPUESTAS ESPERADAS (verificar manualmente):
print()
print("¿Son correctas?")
assert mi_depth(T, A) == 0, "depth(A) debe ser 0"
assert mi_depth(T, B) == 1, "depth(B) debe ser 1"
assert mi_depth(T, E) == 2, "depth(E) debe ser 2"
assert mi_depth(T, I) == 3, "depth(I) debe ser 3"
print("  ✅ depth() verificado")

---

## 📝 Ejercicio 2: Implementación de `height()`

Hay dos versiones:
- **height1** O(n²): `max(depth(p) for p in leaves)`
- **height2** O(n): recursiva, de abajo hacia arriba

Implementar **ambas** y comparar resultados.

In [ ]:
# Ejercicio 2 — height1 y height2

def height1(T, p=None):
    """Altura del subárbol con raíz p — versión O(n²).
    COMPLETAR: usar mi_depth sobre las hojas.
    Pista: max(mi_depth(p) for p en hojas del subárbol) - mi_depth(root_subárbol)
    """
    if p is None: p = T.root()
    # Las hojas del subárbol  son las posiciones del subárbol que son hojas
    def subtree_positions(T, node):
        yield node
        for c in T.children(node):
            yield from subtree_positions(T, c)
    base_depth = mi_depth(T, p)
    leaf_depths = [mi_depth(T, q) for q in subtree_positions(T, p) if T.is_leaf(q)]
    return max(leaf_depths) - base_depth


def height2(T, p=None):
    """Altura del subárbol con raíz p — versión O(n).
    COMPLETAR: recursión bottom-up.
    height(hoja) = 0
    height(p) = 1 + max(height(c) for c in children(p))
    """
    if p is None: p = T.root()
    if T.is_leaf(p):
        return 0
    else:
        return 1 + max(height2(T, c) for c in T.children(p))


# Verificar que ambas dan el mismo resultado
print("=== Comparación height1 vs height2 ===")
for nombre, pos in [('A',A),('B',B),('D',D)]:
    h1 = height1(T, pos)
    h2 = height2(T, pos)
    match = '✅' if h1 == h2 else '❌'
    print(f"  height({nombre}): height1={h1}, height2={h2}  {match}")

print()
print(f"Altura total del árbol (height1): {height1(T)}")
print(f"Altura total del árbol (height2): {height2(T)}")

---

## 📝 Ejercicio 3: Simulación de recorridos

Dado el árbol binario a continuación, **trazar** (simular paso a paso) cada recorrido.
```
        ___1___
       /       \
      2         3
     / \       /
    4   5     6
           / \
          7   8
```
Predecir el orden de visita para: preorden, postorden, inorden, BFS.

In [ ]:
from abc import ABCMeta, abstractmethod

# Árbol binario simple para ejercicio 3
class BinaryTree(Tree):
    @abstractmethod
    def left(self,p): raise NotImplementedError
    @abstractmethod
    def right(self,p): raise NotImplementedError
    def sibling(self,p):
        par=self.parent(p)
        if par is None: return None
        return self.right(par) if p==self.left(par) else self.left(par)
    def children(self,p):
        if self.left(p): yield self.left(p)
        if self.right(p): yield self.right(p)
    def num_children(self,p):
        return (self.left(p) is not None)+(self.right(p) is not None)

class LinkedBinaryTree(BinaryTree):
    class _Node:
        __slots__='_e','_parent','_left','_right'
        def __init__(self,e,parent=None,left=None,right=None):
            self._e=e; self._parent=parent; self._left=left; self._right=right
    class _Position(BinaryTree.Position):
        def __init__(self,c,n): self._container=c; self._node=n
        def element(self): return self._node._e
        def __eq__(self,other): return type(other) is type(self) and other._node is self._node
    def _validate(self,p):
        if not isinstance(p,self._Position): raise TypeError
        if p._container is not self: raise ValueError
        if p._node._parent is p._node: raise ValueError('deprecated')
        return p._node
    def _make_position(self,n): return self._Position(self,n) if n else None
    def __init__(self): self._root=None; self._size=0
    def __len__(self): return self._size
    def root(self): return self._make_position(self._root)
    def parent(self,p): return self._make_position(self._validate(p)._parent)
    def left(self,p): return self._make_position(self._validate(p)._left)
    def right(self,p): return self._make_position(self._validate(p)._right)
    def add_root(self,e):
        self._root=self._Node(e); self._size=1; return self._make_position(self._root)
    def add_left(self,p,e):
        n=self._validate(p)
        if n._left: raise ValueError
        n._left=self._Node(e,parent=n); self._size+=1; return self._make_position(n._left)
    def add_right(self,p,e):
        n=self._validate(p)
        if n._right: raise ValueError
        n._right=self._Node(e,parent=n); self._size+=1; return self._make_position(n._right)
    def inorder(self):
        if not self.is_empty():
            yield from self._subtree_inorder(self.root())
    def _subtree_inorder(self,p):
        if self.left(p): yield from self._subtree_inorder(self.left(p))
        yield p
        if self.right(p): yield from self._subtree_inorder(self.right(p))

# Construir árbol del diagrama
BT = LinkedBinaryTree()
n1 = BT.add_root(1)
n2 = BT.add_left(n1,2); n3 = BT.add_right(n1,3)
n4 = BT.add_left(n2,4); n5 = BT.add_right(n2,5)
n6 = BT.add_left(n3,6)
n7 = BT.add_left(n6,7); n8 = BT.add_right(n6,8)

print("=== Ejercicio 3: Recorridos del árbol binario ===")
print()

# Preorden
def preorder(T,p=None):
    if p is None: p=T.root()
    yield p.element()
    for c in T.children(p): yield from preorder(T,c)

# Postorden
def postorder(T,p=None):
    if p is None: p=T.root()
    for c in T.children(p): yield from postorder(T,c)
    yield p.element()

# BFS
def bfs(T):
    if T.is_empty(): return
    q=deque([T.root()])
    while q:
        p=q.popleft(); yield p.element()
        for c in T.children(p): q.append(c)

print(f"Preorden:  {list(preorder(BT))}")
print(f"Postorden: {list(postorder(BT))}")
print(f"Inorden:   {[p.element() for p in BT.inorder()]}")
print(f"BFS:       {list(bfs(BT))}")
print()
print("=== Verificación ===")
assert list(preorder(BT))  == [1,2,4,5,3,6,7,8], "preorden incorrecto"
assert list(postorder(BT)) == [4,5,2,7,8,6,3,1], "postorden incorrecto"
assert [p.element() for p in BT.inorder()] == [4,2,5,1,7,6,8,3], "inorden incorrecto"
assert list(bfs(BT))       == [1,2,3,4,5,6,7,8], "BFS incorrecto"
print("✅ Todos los recorridos correctos")

---

## 📝 Ejercicio 4: Implementar BFS desde cero (sin ver código anterior)

Implementar la función `breadth_first_traversal(T)` que:
1. Usa una `deque` como cola.
2. Retorna una lista con los elementos visitados en orden BFS.
3. Muestra también el nivel de cada elemento.

In [ ]:
def breadth_first_traversal(T):
    """Retorna lista de (nivel, elemento) en orden BFS.
    COMPLETAR: Implementar el algoritmo de BFS.
    Pistas:
      - Encolar la raíz con nivel 0
      - Mientras la cola no esté vacía:
          - Desencolar (p, nivel)
          - Registrar (nivel, p.element())
          - Encolar cada hijo de p con (nivel+1)
    """
    result = []
    if T.is_empty():
        return result
    q = deque()
    q.append((T.root(), 0))                   # (posición, nivel)
    while q:
        p, nivel = q.popleft()
        result.append((nivel, p.element()))
        for child in T.children(p):
            q.append((child, nivel + 1))
    return result

print("=== BFS con niveles ===")
visitas = breadth_first_traversal(BT)
nivel_actual = -1
for nivel, elem in visitas:
    if nivel != nivel_actual:
        print(f"  Nivel {nivel}:", end=" ")
        nivel_actual = nivel
    print(elem, end=" ")
    if nivel_actual < nivel:
        print()
print()

# Verificar orden
elements_only = [e for _, e in visitas]
assert elements_only == [1,2,3,4,5,6,7,8], f"BFS incorrecto: {elements_only}"
print("✅ BFS implementado correctamente")

---

## 📝 Ejercicio 5: Contador de hojas e internos

Implementar funciones para contar nodos hoja e internos, y verificar la Prop. 8.10.

In [ ]:
def count_external(T, p=None):
    """Cuenta nodos hoja (externos) en el subárbol de p.
    COMPLETAR: contar recursivamente.
    """
    if p is None: p = T.root()
    if T.is_leaf(p):
        return 1
    return sum(count_external(T, c) for c in T.children(p))

def count_internal(T, p=None):
    """Cuenta nodos internos en el subárbol de p."""
    if p is None: p = T.root()
    if T.is_leaf(p):
        return 0
    return 1 + sum(count_internal(T, c) for c in T.children(p))

def check_proper(T, p=None):
    """Retorna True si el árbol binario es propio (cada nodo es hoja o tiene 2 hijos)."""
    if p is None: p = T.root()
    if T.is_leaf(p):
        return True
    if T.num_children(p) == 1:
        return False
    return all(check_proper(T, c) for c in T.children(p))

# Verificar sobre el árbol del ejercicio 3
nE = count_external(BT)
nI = count_internal(BT)
es_propio = check_proper(BT)
h = height2(BT)

print("=== Estadísticas del árbol binario ===")
print(f"  Nodos externos (hojas): {nE}")
print(f"  Nodos internos:         {nI}")
print(f"  Total:                  {nE + nI}")
print(f"  Altura:                 {h}")
print(f"  ¿Es árbol propio?:      {es_propio}")
print()
if es_propio:
    print("  Verificando nE = nI + 1 (Prop 8.10):")
    print(f"    {nE} = {nI} + 1  →  {'✅' if nE == nI + 1 else '❌'}")
else:
    print("  (Prop 8.10 solo aplica para árboles propios)")
print()

# Árbol propio de prueba
BT_propio = LinkedBinaryTree()
rp = BT_propio.add_root('*')
lp = BT_propio.add_left(rp,'+')
rp2 = BT_propio.add_right(rp,'-')
BT_propio.add_left(lp,'3'); BT_propio.add_right(lp,'1')
BT_propio.add_left(rp2,'9'); BT_propio.add_right(rp2,'5')
nE2 = count_external(BT_propio); nI2 = count_internal(BT_propio)
print("Árbol propio (3+1)*(9-5):")
print(f"  nE={nE2}, nI={nI2}, ¿propio? {check_proper(BT_propio)}")
print(f"  nE = nI + 1: {nE2} = {nI2} + 1  →  {'✅' if nE2 == nI2+1 else '❌'}")

---

## 🏆 Desafío: Imprimir árbol por niveles como tabla

Implementar `print_by_levels(T)` que imprime el árbol nivel por nivel, mostrando
los elementos de cada nivel en una fila separada con su número de nivel.

In [ ]:
def print_by_levels(T):
    """Imprime los nodos del árbol agrupados por nivel.
    COMPLETAR: usar BFS para agrupar por nivel.
    """
    if T.is_empty():
        print("(árbol vacío)")
        return
    current = [T.root()]
    level = 0
    while current:
        elems = [p.element() for p in current]
        width = max(len(str(e)) for e in elems)
        row = "  ".join(f"{str(e):^{width}}" for e in elems)
        print(f"Nivel {level}: {row}")
        next_level = []
        for p in current:
            for c in T.children(p):
                next_level.append(c)
        current = next_level
        level += 1

print("=== Árbol del Ejercicio 3 por niveles ===")
print_by_levels(BT)
print()
print("=== Árbol libro por niveles ===")
print_by_levels(T)

---

## ✅ Resumen de lo practicado

| Ejercicio | Concepto | Complejidad |
|-----------|----------|-------------|
| Ej. 1 | depth() recursivo | O(profundidad) |
| Ej. 2 | height1 vs height2 | O(n²) vs O(n) |
| Ej. 3 | 4 recorridos | todos O(n) |
| Ej. 4 | BFS con deque | O(n) |
| Ej. 5 | Propiedades BT | O(n) |
| Desafío | Print por niveles | O(n) |

## 📚 Referencias

- Goodrich et al., §8.1–§8.4
- Algoritmos de recorrido: §8.4.1–§8.4.4

---
*Capítulo 8 — Árboles | Estructuras de Datos y Algoritmos — Lic. en Sistemas*  
*Material bajo licencia [CC BY-SA 4.0](https://creativecommons.org/licenses/by-sa/4.0/)*